# Assignment 4: End-to-end, raw file to model-ready data

**Dataset:** Adult (Census Income), UCI Machine Learning Repository
**Source link:** https://archive.ics.uci.edu/dataset/2/adult
**Direct file used:** https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data
(backup mirror: https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv)

The raw file has 15 columns and more than 32,000 rows, mixes numeric and text columns, and has genuine
missing values (marked `?`). The target is whether a person earns more than 50K a year.

**No model is trained here.** The notebook stops at a clean matrix.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.feature_selection import VarianceThreshold

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)
RANDOM_STATE = 42

## Function 1: `load_data()`
Reads the file exactly as published. The only thing it does is tell pandas that `?` means missing.
Everything else (stray spaces, duplicates, odd values) is left for the quality report.

In [ ]:
URLS = [
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv",
]

COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income",
]


def load_data():
    """Read the raw Adult file with no cleaning. Tries each URL until one works."""
    for url in URLS:
        try:
            df = pd.read_csv(url, header=None, names=COLUMNS, na_values=["?", " ?"])
            print("loaded from:", url)
            return df
        except Exception as e:
            print("could not load", url, "->", e)
    raise RuntimeError("None of the data sources worked. Check your internet connection.")

## Question 1: Data dictionary

| Column | Meaning | Type | Unit | Expected range |
|---|---|---|---|---|
| `age` | Age of the person | integer | years | 17 to 90 |
| `workclass` | Type of employer | category | n/a | 8 values (Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked) |
| `fnlwgt` | Census sampling weight (how many people this row stands for) | integer | people | greater than 0 (about 12,000 to 1,500,000) |
| `education` | Highest education level reached | category | n/a | 16 levels, Preschool to Doctorate |
| `education_num` | Numeric code of `education` | integer (ordinal) | level | 1 to 16 |
| `marital_status` | Marital status | category | n/a | 7 values |
| `occupation` | Kind of job | category | n/a | 14 values |
| `relationship` | Role in the household | category | n/a | 6 values |
| `race` | Race | category | n/a | 5 values |
| `sex` | Sex | category | n/a | Male, Female |
| `capital_gain` | Income from investments and property sales | integer | US dollars per year | 0 to 99,999 (99,999 is a top-coded cap) |
| `capital_loss` | Losses from investments and property sales | integer | US dollars per year | 0 to about 4,400 |
| `hours_per_week` | Hours worked per week | integer | hours | 1 to 99 |
| `native_country` | Country of origin | category | n/a | about 41 countries |
| `income` | **Target.** Yearly income band | category | US dollars per year | `<=50K` or `>50K` |

## Question 2: Raw quality report
Every check below runs on the untouched raw data.

In [ ]:
raw = load_data()
print("raw shape:", raw.shape)
raw.head()

In [ ]:
text_cols = raw.select_dtypes(include=["object", "string"]).columns
problems = []   # (problem, count)

# 1. missing values, one row per affected column
for col, n in raw.isna().sum().items():
    if n > 0:
        problems.append((f"missing values in {col} ({n / len(raw):.1%})", int(n)))

# 2. duplicates
problems.append(("exact duplicate rows", int(raw.duplicated().sum())))

# 3. impossible values
problems.append(("age below 0 or above 100", int(((raw.age < 0) | (raw.age > 100)).sum())))
problems.append(("hours_per_week at or below 0", int((raw.hours_per_week <= 0).sum())))
problems.append(("negative capital_gain or capital_loss", int(((raw.capital_gain < 0) | (raw.capital_loss < 0)).sum())))
problems.append(("education_num outside 1 to 16", int((~raw.education_num.between(1, 16)).sum())))
problems.append(("fnlwgt at or below 0", int((raw.fnlwgt <= 0).sum())))

# 4. suspicious placeholder value
problems.append(("capital_gain equal to 99999 (top-coded placeholder)", int((raw.capital_gain == 99999).sum())))

# 5. inconsistent units (hours per week above 168 would mean a different unit was used)
problems.append(("hours_per_week above 168 (impossible, wrong unit?)", int((raw.hours_per_week > 168).sum())))

# 6. inconsistent category labels
padded = sum(int((raw[c].dropna() != raw[c].dropna().str.strip()).sum()) for c in text_cols)
problems.append(("text cells with leading or trailing spaces", padded))

variants = sum(raw[c].nunique() - raw[c].str.strip().str.lower().str.rstrip(".").nunique() for c in text_cols)
problems.append(("labels that are the same category written differently", int(variants)))

conflicts = int((raw.groupby("education")["education_num"].nunique() > 1).sum())
problems.append(("education labels mapped to more than one education_num", conflicts))

quality = pd.DataFrame(problems, columns=["problem", "count"])
quality

In [ ]:
print("Target labels as stored in the raw file:", raw["income"].unique())
print()
print(raw["income"].str.strip().value_counts(normalize=True).round(3))

## Question 3: New features

| Feature | How it is built | Why |
|---|---|---|
| `net_capital` | `capital_gain - capital_loss` | A person usually has a gain or a loss, rarely both, so one number says whether investments made or lost money. |
| `potential_experience` | `age - education_num - 5`, never below 0 | Roughly the years since schooling ended, which is a better proxy for work experience than age alone. |
| `is_married` | 1 if marital status is a married-with-spouse value | Two-income households tend to sit in the higher band, and this collapses 7 categories into the one signal that matters. |
| `hours_bracket` | part_time (under 35), full_time (35 to 45), overtime (over 45) | The effect of hours is not a straight line, so bands capture it better than the raw number. |
| `gain_topcoded` | 1 if `capital_gain == 99999` | That value is a cap, not a real amount, so a flag lets a model treat those rows separately. |

## Function 2: `clean(df)`
Only fixes things that do not depend on the data split: whitespace, labels, duplicates, impossible values and columns we do not want.
Anything that has to be *learned* from data (imputation values, scaling, encoding) is left to the pipeline so it can be fitted on train only.

In [ ]:
def clean(df):
    """Fix the raw problems found in the quality report. Does not learn anything from the data."""
    df = df.copy()

    # whitespace and the '?' marker, in every text column
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = df[col].str.strip().replace("?", np.nan)

    # target: one standard label set, then 0/1
    df["income"] = df["income"].str.rstrip(".")
    df["income"] = (df["income"] == ">50K").astype(int)

    # exact duplicates
    df = df.drop_duplicates()

    # impossible values: drop those rows
    bad = (
        (df.age < 0) | (df.age > 100)
        | (df.hours_per_week <= 0) | (df.hours_per_week > 168)
        | (df.capital_gain < 0) | (df.capital_loss < 0)
        | (df.education_num < 1) | (df.education_num > 16)
        | (df.fnlwgt <= 0)
    )
    df = df[~bad]

    # education is the same information as education_num (checked in the quality report)
    # fnlwgt is a survey sampling weight, not a property of the person
    df = df.drop(columns=["education", "fnlwgt"])

    # any column that is more than half empty is not worth keeping
    too_empty = df.columns[df.isna().mean() > 0.5]
    df = df.drop(columns=too_empty)

    return df.reset_index(drop=True)

## Function 3: `add_features(df)`

In [ ]:
def add_features(df):
    """Add the five features described in Question 3."""
    df = df.copy()
    df["net_capital"] = df["capital_gain"] - df["capital_loss"]
    df["potential_experience"] = (df["age"] - df["education_num"] - 5).clip(lower=0)
    df["is_married"] = df["marital_status"].isin(["Married-civ-spouse", "Married-AF-spouse"]).astype(int)
    df["hours_bracket"] = pd.cut(
        df["hours_per_week"], bins=[0, 34, 45, 200], labels=["part_time", "full_time", "overtime"]
    ).astype(object)
    df["gain_topcoded"] = (df["capital_gain"] == 99999).astype(int)
    return df

## Function 4: `build_pipeline()`
Returns an **unfitted** pipeline. It is fitted on the training rows only, inside the final cell.

* skewed money columns: median impute, then Yeo-Johnson (also scales)
* other numbers: median impute, then standard scaling
* text columns: fill missing with `"Unknown"`, then one-hot encode. Unseen categories are ignored and categories under 1% are grouped together
* 0/1 flags: passed through
* a last step removes any column that ends up constant

In [ ]:
def build_pipeline():
    skewed = ["capital_gain", "capital_loss", "net_capital"]
    numeric = ["age", "education_num", "hours_per_week", "potential_experience"]
    flags = ["is_married", "gain_topcoded"]
    categorical = ["workclass", "marital_status", "occupation", "relationship",
                   "race", "sex", "native_country", "hours_bracket"]

    skewed_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("power", PowerTransformer(method="yeo-johnson")),
    ])
    numeric_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_steps = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=0.01, sparse_output=False)),
    ])

    columns = ColumnTransformer([
        ("skewed", skewed_steps, skewed),
        ("numeric", numeric_steps, numeric),
        ("flags", "passthrough", flags),
        ("categorical", categorical_steps, categorical),
    ])

    pipe = Pipeline([
        ("columns", columns),
        ("drop_constant", VarianceThreshold(threshold=0.0)),
    ])
    pipe.set_output(transform="pandas")
    return pipe

## Question 5: Full path, raw file to `X_train`, `X_test`, `y_train`, `y_test`
The split happens **before** the pipeline is fitted, so nothing is learned from the test rows.

In [ ]:
raw_df = load_data()
clean_df = clean(raw_df)
featured_df = add_features(clean_df)

X = featured_df.drop(columns="income")
y = featured_df["income"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

pipeline = build_pipeline()
X_train = pipeline.fit_transform(X_train_raw)   # fit on train only
X_test = pipeline.transform(X_test_raw)         # test is only transformed

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print()
print("share of income > 50K  train:", round(y_train.mean(), 3), " test:", round(y_test.mean(), 3))

## Question 6: Proof that the output is clean

In [ ]:
assert X_train.isna().sum().sum() == 0, "NaN left in X_train"
assert X_test.isna().sum().sum() == 0, "NaN left in X_test"
assert X_train.shape[1] == X_test.shape[1], "train and test have different column counts"
assert list(X_train.columns) == list(X_test.columns), "train and test columns differ"
assert (X_train.nunique() > 1).all(), "a constant column is present"
assert len(X_train) == len(y_train) and len(X_test) == len(y_test), "rows and labels do not line up"
print("All checks passed.")

## Question 7: Before and after

In [ ]:
final_all = pd.concat([X_train, X_test])

before_after = pd.DataFrame({
    "stage": ["raw", "final"],
    "rows": [raw_df.shape[0], final_all.shape[0]],
    "columns": [raw_df.shape[1], final_all.shape[1]],
    "missing_values": [int(raw_df.isna().sum().sum()), int(final_all.isna().sum().sum())],
    "duplicate_rows": [int(raw_df.duplicated().sum()), int(final_all.duplicated().sum())],
}).set_index("stage")
display(before_after)

# ready-to-paste markdown version for the README
lines = ["| Stage | Rows | Columns | Missing values | Duplicate rows |", "|---|---|---|---|---|"]
for stage, r in before_after.iterrows():
    lines.append(f"| {stage} | {r.rows:,} | {r.columns} | {r.missing_values:,} | {r.duplicate_rows:,} |")
print("\n".join(lines))

The exact duplicates of the raw file were removed in `clean()`. If the final-stage duplicate count above is not zero, those are rows that look identical
*after* `fnlwgt` and `education` were dropped and the columns were scaled and grouped. They are different people who share the same
attributes, so they are real observations and are kept on purpose.

## Question 8: Save and download

In [ ]:
featured_df.to_csv("adult_clean.csv", index=False)
joblib.dump(pipeline, "pipeline.joblib")

# check that the saved pipeline loads and gives the same numbers
loaded = joblib.load("pipeline.joblib")
assert np.allclose(loaded.transform(X_test_raw.head()).values, X_test.head().values)
print("saved adult_clean.csv and pipeline.joblib, reload check passed")

try:
    from google.colab import files
    files.download("adult_clean.csv")
    files.download("pipeline.joblib")
except ImportError:
    print("Not running in Colab, files are in the current folder.")

## Question 9: Every decision and the number behind it

In [ ]:
n_dup = int(raw_df.duplicated().sum())
miss = raw_df.isna().sum()
topcoded = int((raw_df.capital_gain == 99999).sum())
rows_dropped = len(raw_df) - len(clean_df)
n_final_cols = X_train.shape[1]

decisions = f"""
* Removed **{n_dup}** exact duplicate rows, because the same record twice would count double.
* Dropped **{len(raw_df) - n_dup - len(clean_df)}** rows that failed the impossible-value checks (age, hours, capital, education_num, fnlwgt).
* Total rows removed by `clean()`: **{rows_dropped}** out of {len(raw_df):,}.
* Dropped `education`, because it maps one-to-one onto `education_num` ({conflicts} conflicting labels found), so it carries no extra information.
* Dropped `fnlwgt`, because it is a census sampling weight, not something about the person.
* Kept `workclass` ({int(miss['workclass'])} missing, {miss['workclass'] / len(raw_df):.1%}), `occupation` ({int(miss['occupation'])} missing, {miss['occupation'] / len(raw_df):.1%}) and `native_country` ({int(miss['native_country'])} missing, {miss['native_country'] / len(raw_df):.1%}): all are far below the 50% limit used to drop a column.
* Filled those missing text values with the label `Unknown` instead of the most common value, because missing `workclass` and `occupation` tend to occur together and are informative.
* Numeric columns use median imputation (the median is not pulled by the long tails). No numeric column had missing values in this file, but the pipeline is ready if new data does.
* Did not delete high `hours_per_week` or high `age` rows: the maximums are within the possible range, so they are rare but real.
* Kept the **{topcoded}** rows with `capital_gain = 99999` and added the `gain_topcoded` flag instead of deleting them.
* Applied Yeo-Johnson to `capital_gain`, `capital_loss` and `net_capital`, because most values are 0 with a few very large ones.
* One-hot encoded the text columns and grouped any category under 1% of the training rows, which keeps the matrix at **{n_final_cols}** columns instead of one column per rare country.
* Split 80/20, stratified on `income`, `random_state={RANDOM_STATE}`, **before** fitting anything, because the target is imbalanced ({y.mean():.1%} earn more than 50K).
"""
display(Markdown(decisions))

## Question 10: What is still wrong with this dataset

The data comes from the 1994 US census, so income and job patterns are decades old and a model built on it would not describe people today.
The `>50K` threshold is in 1994 dollars and is never adjusted. The `?` values in `workclass`, `occupation` and `native_country` are not missing at random
(for example, people without a job have no occupation), so filling them with `Unknown` is a sensible choice but does not recover the real values.
The value 99,999 in `capital_gain` hides the true amounts and cannot be repaired. The target is imbalanced, with roughly three people in four in the lower band,
and the file contains sensitive attributes (`race`, `sex`, `native_country`) that can lead a model to treat groups unfairly. Finally, many rows still share identical
feature values after cleaning, which is a limit of how coarse the columns are, not something cleaning can fix.